# Laboratorium 6: projekt generacyjny cz. 1, przygotowanie danych

Autor: Adam Pastuszak

Uruchomienie: Runtime → Run all, notatnik pobiera tekst z Wolnych Lektur

## 1. Importy

In [ ]:
import urllib.request

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(0)

## 2. Korpus: Pan Tadeusz z Wolnych Lektur

Wybrałem Pana Tadeusza Adama Mickiewicza (domena publiczna), inny tekst niż Lalka
z ćwiczenia, z tekstu usuwam stopkę Wolnych Lektur (wszystko od linii z kreskami)
i zamieniam końce linii Windows \r\n na \n, żeby \r nie był osobnym znakiem

In [ ]:
URL = "https://wolnelektury.pl/media/book/txt/pan-tadeusz.txt"
surowy = urllib.request.urlopen(URL).read().decode("utf-8")
tekst = surowy.replace("\r\n", "\n").replace("\r", "\n")
stopka = tekst.find("\n-----")
if stopka > 0:
    tekst = tekst[:stopka]
tekst = tekst.strip() + "\n"
print("Znaków przed czyszczeniem:", len(surowy))
print("Znaków po czyszczeniu    :", len(tekst))
print("Początek:", repr(tekst[:200]))
print("Koniec  :", repr(tekst[-150:]))

## 3. Słownik znaków i kodowanie

In [ ]:
znaki = sorted(set(tekst))
stoi = {z: i for i, z in enumerate(znaki)}
itos = {i: z for z, i in stoi.items()}
V = len(znaki)


def enc(s):
    return [stoi[c] for c in s]


def dec(ids):
    return "".join(itos[i] for i in ids)


dane = np.array(enc(tekst), dtype=np.int64)
assert dec(dane[:500].tolist()) == tekst[:500]
print("Rozmiar słownika (unikalne znaki):", V)
print("Znaki:", repr("".join(znaki)))
print("Zakodowany tekst, kształt:", dane.shape)
print("Przykład kodowania 'Litwo':", enc("Litwo"))

## 4. Podział trening / walidacja i Dataset

Pierwsze 90% tekstu to trening, ostatnie 10% walidacja (ciągłe kawałki, żeby okna
z walidacji nie powtarzały się w treningu), wejście to okno SEKW znaków,
cel to to samo okno przesunięte o 1 znak

In [ ]:
class DatasetZnakowy(Dataset):
    def __init__(self, dane, sekw):
        self.dane = dane
        self.sekw = sekw

    def __len__(self):
        return len(self.dane) - self.sekw

    def __getitem__(self, i):
        x = self.dane[i:i + self.sekw]
        y = self.dane[i + 1:i + self.sekw + 1]
        return torch.tensor(x), torch.tensor(y)


podzial = int(0.9 * len(dane))
dane_tr, dane_val = dane[:podzial], dane[podzial:]
SEKW = 40
BATCH = 32
ds = DatasetZnakowy(dane_tr, SEKW)
ds_val = DatasetZnakowy(dane_val, SEKW)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True,
                generator=torch.Generator().manual_seed(0))
print("Znaki trening / walidacja:", len(dane_tr), "/", len(dane_val))
print("Przykłady trening / walidacja:", len(ds), "/", len(ds_val))

## 5. Sprawdzenie batcha

In [ ]:
xb, yb = next(iter(dl))
print("Batch X:", tuple(xb.shape), " Y:", tuple(yb.shape))
for k in range(3):
    print(f"--- przykład {k}")
    print("Wejście:", repr(dec(xb[k].tolist())))
    print("Cel    :", repr(dec(yb[k].tolist())))
assert torch.equal(xb[:, 1:], yb[:, :-1])
pary = zip(xb[0][:5].tolist(), yb[0][:5].tolist())
print("Pierwsze 5 par (wejście → cel):",
      [(dec([a]), dec([b])) for a, b in pary])
print("Przesunięcie poprawne w całym batchu (cel = wejście + 1)")

## 6. Statystyki zbioru

In [ ]:
print(f"Znaków w korpusie       : {len(tekst)}")
print(f"Unikalnych znaków       : {V}")
print(f"Długość okna SEKW       : {SEKW}")
print(f"Przykłady (trening)     : {len(ds)}")
print(f"Przykłady (walidacja)   : {len(ds_val)}")
print(f"Batchy na epokę (b={BATCH}) : {len(dl)}")

## 7. Wpływ długości okna SEKW na liczbę przykładów

Przy oknach co 1 znak liczba przykładów to N - SEKW, dla porównania liczę też
okna bez nakładania (co SEKW + 1 znaków) i ile znaków celu przypada na epokę

In [ ]:
N = len(dane_tr)
print(f"{'SEKW':>5} | {'przykłady':>9} | {'batche':>6} | "
      f"{'znaki celu na epokę':>20} | {'okna bez nakładania':>19}")
for s in [10, 20, 40, 80, 160, 320]:
    n = len(DatasetZnakowy(dane_tr, s))
    print(f"{s:>5} | {n:>9} | {n // BATCH:>6} | {n * s:>20} | "
          f"{N // (s + 1):>19}")

## Sprawozdanie

### 1. Cel ćwiczenia

Celem było przygotować dane do modelu Char-RNN na własnym polskim tekście:
słownik znaków, kodowanie tekstu na liczby, okna wejście-cel przesunięte o 1 znak i DataLoader

Dane wykorzystam na zajęciach 7 do uczenia modelu

### 2. Metoda i implementacja

Narzędzia: Python, numpy, PyTorch (Dataset, DataLoader)

Korpus: Pan Tadeusz Adama Mickiewicza z Wolnych Lektur (domena publiczna), wybrałem inny tekst niż Lalka z ćwiczenia

Przygotowanie tekstu: usunąłem stopkę Wolnych Lektur z informacją o licencji i zamieniłem końce linii \r\n na \n, dzięki temu \r nie zajmuje miejsca w słowniku

Słownik to posortowane unikalne znaki tekstu, stoi zamienia znak na numer, a itos numer na znak, sprawdziłem, że dekodowanie zakodowanego tekstu daje ten sam tekst

Podział: pierwsze 90% tekstu to trening, ostatnie 10% walidacja, jako dwa ciągłe kawałki, żeby okna z walidacji nie nachodziły na okna z treningu

Przykład to okno SEKW = 40 znaków, a cel to to samo okno przesunięte o 1 znak, batch 32, tasowanie z ziarnem 0

### 3. Wyniki

Rozmiar korpusu, liczba unikalnych znaków i liczba przykładów są w punktach 2, 3, 4 i 6 notatnika

Słownik zawiera małe i duże litery z polskimi znakami, cyfry, spację, znak nowej linii i interpunkcję, w tym znaki typowe dla tego wydania, jak myślnik i cudzysłowy

Zdekodowany batch jest w punkcie 5, wejście i cel to ten sam fragment tekstu, cel zaczyna się o jeden znak później i ma jeden znak więcej na końcu

Przesunięcie sprawdziłem asercją dla całego batcha (wszystkie 32 przykłady), nie tylko dla pierwszego

Wpływ SEKW na liczbę przykładów jest w tabeli w punkcie 7

### 4. Analiza i interpretacja

Przy oknach przesuwanych co 1 znak liczba przykładów to N - SEKW, więc prawie nie zależy od długości okna, wydłużenie okna z 10 do 320 zmniejsza ją tylko o 310 przykładów

Za to rośnie liczba znaków celu w jednej epoce (przykłady razy SEKW), bo każdy znak tekstu trafia do wielu nakładających się okien, przy SEKW = 320 epoka jest około 32 razy dłuższa niż przy SEKW = 10

Kolejne okna różnią się tylko jednym znakiem, więc przykłady są bardzo do siebie podobne i liczba przykładów zawyża ilość naprawdę różnych danych

Okna bez nakładania pokazują drugą skrajność, wtedy liczba przykładów spada mniej więcej jak N / (SEKW + 1), czyli przy dwa razy dłuższym oknie jest prawie dwa razy mniej przykładów

Dłuższe okno daje modelowi więcej kontekstu (przy SEKW = 40 to kilka słów, przy 320 kilka wersów), ale uczenie jest wolniejsze, a RNN i tak słabo wykorzystuje bardzo odległe znaki

Pan Tadeusz jest wierszem, więc znak nowej linii pojawia się regularnie po każdym wersie i model będzie mógł nauczyć się długości wersu i rymów

### 5. Wnioski

Dane do Char-RNN są gotowe: tekst jest zakodowany na liczby, okna wejście-cel są dobrze przesunięte, co sprawdziłem na całym batchu

Usunięcie stopki i znaku \r zmniejsza słownik i nie uczy modelu tekstu, który nie należy do książki

Długość okna przy przesuwaniu co 1 znak prawie nie zmienia liczby przykładów, ale mocno zmienia długość epoki i ilość kontekstu, na zajęciach 7 sensowne jest zacząć od SEKW około 40-100